# 💳 Credit Card Fraud Detection — Classification at Scale

**Session 4 | Industrial Use Case 3 | DSA & ML for Business**

---

### Business Context
- Global credit card fraud losses: **$32 billion/year** (Nilson Report)
- Visa processes **65,000 transactions/second** — each needs real-time fraud scoring
- Missing a fraud (False Negative) costs the **full transaction amount + chargeback fees**
- False alerts (False Positives) cost **$50-100 per investigation**
- Only **~2-8%** of transactions are fraudulent → severe class imbalance

### What You'll Learn
1. **EDA** on fraud patterns — time-of-day, amount, distance signals
2. Train **Decision Tree, Random Forest, SGD Classifier**
3. Handle **class imbalance** with balanced class weights
4. Apply **cost matrix** — quantify dollar impact of each error type
5. **Threshold optimization** to minimize total fraud losses

## Step 1: Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score
import warnings; warnings.filterwarnings('ignore')

df = pd.read_csv("../datasets/credit_card_fraud.csv")
print(f"Dataset: {df.shape[0]} transactions")
print(f"Fraud rate: {df['is_fraud'].mean():.2%}")
print(f"\nFraud vs Legit amounts:")
print(df.groupby('is_fraud')['amount'].describe().round(2))
df.head()

## Step 2: EDA — Fraud Patterns

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, feat in zip(axes.flat[:5], ['amount', 'hour_of_day', 'distance_from_home_km', 'transactions_last_24h', 'is_foreign_transaction']):
    for lbl, color, name in [(0, '#10B981', 'Legit'), (1, '#EF4444', 'Fraud')]:
        ax.hist(df[df['is_fraud']==lbl][feat], bins=30, alpha=0.5, label=name, color=color, density=True)
    ax.set_title(feat.replace('_', ' ').title(), fontweight='bold'); ax.legend(fontsize=8)
fraud_by_cat = df.groupby('merchant_category')['is_fraud'].mean().sort_values(ascending=False)
axes[1, 2].barh(fraud_by_cat.index, fraud_by_cat.values, color='#EF4444')
axes[1, 2].set_title('Fraud Rate by Merchant', fontweight='bold')
plt.suptitle('Fraud Pattern Analysis', fontsize=15, fontweight='bold')
plt.tight_layout(); plt.show()

## Step 3: Train Models & Compare Performance

In [ ]:
le = LabelEncoder()
df['merchant_encoded'] = le.fit_transform(df['merchant_category'])
feature_cols = ['amount', 'hour_of_day', 'distance_from_home_km', 'transactions_last_24h',
                'is_foreign_transaction', 'is_online', 'used_chip', 'merchant_encoded']
X = df[feature_cols]; y = df['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train); X_test_sc = scaler.transform(X_test)

models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=6, class_weight='balanced', random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'SGD Classifier': SGDClassifier(loss='log_loss', class_weight='balanced', random_state=42, max_iter=1000),
}
results = {}
for name, model in models.items():
    if 'SGD' in name:
        model.fit(X_train_sc, y_train); yp = model.predict(X_test_sc); yprob = model.decision_function(X_test_sc)
    else:
        model.fit(X_train, y_train); yp = model.predict(X_test); yprob = model.predict_proba(X_test)[:, 1]
    results[name] = {'pred': yp, 'prob': yprob, 'auc': roc_auc_score(y_test, yprob)}
    print(f"\n{'='*50}\n{name} | ROC-AUC: {results[name]['auc']:.4f}\n{'='*50}")
    print(classification_report(y_test, yp, target_names=['Legit', 'Fraud']))

## Step 4: Cost Matrix & Threshold Optimization

**Fraud cost matrix:**
- **False Negative** (missed fraud): average fraud amount + $25 chargeback fee
- **False Positive** (legitimate flagged): $75 investigation cost

In [ ]:
avg_fraud_amount = df[df['is_fraud']==1]['amount'].mean()
COST_FN = avg_fraud_amount + 25
COST_FP = 75
print(f"Avg fraud amount: ${avg_fraud_amount:,.2f}")
print(f"Cost FN: ${COST_FN:,.2f} | Cost FP: ${COST_FP:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['prob'])
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={res['auc']:.3f})", linewidth=2)
axes[0].plot([0,1],[0,1],'k--',alpha=0.3)
axes[0].set_title('ROC Curves', fontweight='bold'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

rf_prob = results['Random Forest']['prob']
thresholds = np.arange(0.1, 0.9, 0.01)
costs = []
for t in thresholds:
    yp = (rf_prob >= t).astype(int)
    cm = confusion_matrix(y_test, yp); tn, fp, fn, tp = cm.ravel()
    costs.append(fn * COST_FN + fp * COST_FP)
opt_t = thresholds[np.argmin(costs)]
axes[1].plot(thresholds, costs, 'b-', linewidth=2)
axes[1].axvline(x=opt_t, color='red', linestyle='--', label=f'Optimal: {opt_t:.2f}')
axes[1].set_title('Cost-Optimized Threshold', fontweight='bold')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

yp_opt = (rf_prob >= opt_t).astype(int)
cm = confusion_matrix(y_test, yp_opt); tn, fp, fn, tp = cm.ravel()
print(f"\nOptimal threshold: {opt_t:.2f}")
print(f"Recall: {tp/(tp+fn):.2%} | Precision: {tp/(tp+fp):.2%}")
print(f"Total cost: ${fn*COST_FN + fp*COST_FP:,.0f}")

daily_txns = 65000 * 86400
scale = daily_txns / len(y_test)
print(f"\n--- At Visa Scale ({daily_txns:,.0f} daily txns) ---")
print(f"Fraud losses prevented: ${tp * COST_FN * scale:,.0f}/day")
print(f"False alarm cost: ${fp * COST_FP * scale:,.0f}/day")